# Funnel Analysis and Drop-Off Detection

Analysing where ride requests are lost in the lifecycle.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.funnel import (
    analyze_funnel,
    compare_funnels,
    compare_high_demand_funnel,
    get_drop_off_points,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

In [ ]:
# Overall funnel
results = analyze_funnel(df)
for stage in results[0].stages:
    print(f'{stage.name}: {stage.count} ({stage.rate:.1%})')

In [ ]:
# Drop-off points
drop_offs = get_drop_off_points(results[0])
for do in drop_offs:
    print(f'{do.from_stage} → {do.to_stage}: {do.count_lost} lost ({do.pct_lost:.1%}), conversion={do.conversion_rate:.1%}')

In [ ]:
# City comparison
city_comp = compare_funnels(df, group_by=['city'])
print(city_comp.to_string())

In [ ]:
# High-demand comparison
demand_comp = compare_high_demand_funnel(df)
print(demand_comp.to_string())